# SDFVD2 GPU training run

This is an **exploratory baseline**, not a validated detector. It reads a larger group-disjoint sample directly from the Drive ZIP with byte-range requests, extracts selected frames only, fine-tunes pretrained ResNet-18, chooses a threshold on validation data, and evaluates once on the held-out test groups. Headline metrics aggregate frame scores by inferred source group. Labels and filename groups are still provisional, so do not present these metrics as final research results.

**Before running:** in Colab select **Runtime → Change runtime type → T4 GPU** (or another GPU). This notebook stops early if CUDA is unavailable. It does not mount Drive or download the full archive.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import os, re, json, random, zipfile, shutil, hashlib, subprocess
from pathlib import Path
from collections import defaultdict

DRIVE_FILE_ID = 'YOUR_SDFVD2_ARCHIVE_FILE_ID'  # SDFVD2.0_v1.zip
WORK = Path('/content/sdfvd2_pilot')
OUT = Path('/content/pilot_training_outputs/sdfvd2_pilot')
GROUPS_PER_CLASS = 48  # use nearly all independent filename groups per class
FRAMES_PER_VIDEO = 8
EPOCHS = 10
SEED = 20261012
WORK.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
import google.auth
from google.auth.transport.requests import AuthorizedSession
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly', 'https://www.googleapis.com/auth/drive.file'])
session = AuthorizedSession(creds)
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
drive_api = build('drive', 'v3', credentials=creds, cache_discovery=False)
meta = session.get(f'https://www.googleapis.com/drive/v3/files/{DRIVE_FILE_ID}', params={'fields':'id,name,size,mimeType','supportsAllDrives':'true'}, timeout=60)
meta.raise_for_status(); meta = meta.json()
assert meta.get('name') == 'SDFVD2.0_v1.zip', f'Unexpected Drive file: {meta}'
print('Drive archive:', meta['name'], f'({int(meta["size"]) / 1e9:.2f} GB)')
class DriveRangeFile:
    BLOCK = 8 * 1024 * 1024
    def __init__(self, file_id, size, session, cache_blocks=3):
        from collections import OrderedDict
        self.file_id, self.size, self.session = file_id, int(size), session
        self.pos, self.cache_blocks, self.cache = 0, cache_blocks, OrderedDict()
    def seekable(self): return True
    def readable(self): return True
    def tell(self): return self.pos
    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        if self.pos < 0: raise ValueError('negative seek position')
        return self.pos
    def _block(self, index):
        if index in self.cache:
            self.cache.move_to_end(index); return self.cache[index]
        start = index * self.BLOCK; end = min(self.size, start + self.BLOCK) - 1
        response = self.session.get(f'https://www.googleapis.com/drive/v3/files/{self.file_id}', params={'alt':'media','supportsAllDrives':'true'}, headers={'Range':f'bytes={start}-{end}'}, timeout=90)
        if response.status_code != 206:
            response.close(); raise RuntimeError(f'Drive did not honor byte-range access (HTTP {response.status_code}); stopping to avoid a full archive download.')
        data = response.content; response.close()
        if len(data) != end - start + 1: raise RuntimeError('Drive returned an incomplete byte range.')
        self.cache[index] = data
        while len(self.cache) > self.cache_blocks: self.cache.popitem(last=False)
        return data
    def read(self, size=-1):
        if size is None or size < 0: size = self.size - self.pos
        size = min(size, self.size - self.pos)
        chunks = []
        while size:
            index, offset = divmod(self.pos, self.BLOCK); data = self._block(index)
            take = min(size, len(data) - offset)
            if take <= 0: break
            chunks.append(data[offset:offset+take]); self.pos += take; size -= take
        return b''.join(chunks)
remote_zip = DriveRangeFile(DRIVE_FILE_ID, meta['size'], session)
print('Connected to Drive with range reads; no archive file is copied to Colab or your laptop.')

In [ ]:
# Read only the ZIP directory, choose a larger group-disjoint sample, and extract only those videos.
with zipfile.ZipFile(remote_zip) as zf:
    members = [m for m in zf.infolist() if not m.is_dir() and m.filename.lower().endswith('.mp4')]
    by_group = defaultdict(list)
    for m in members:
        folders = [p.lower() for p in Path(m.filename.replace('\\', '/')).parts[:-1]]
        has_fake = any(re.search(r'(^|[._-])(fake|forged)$', p) for p in folders)
        has_real = any(re.search(r'(^|[._-])real$', p) for p in folders)
        if has_fake == has_real:
            continue
        label = 1 if has_fake else 0
        stem = Path(m.filename).stem
        group = re.sub(r'(?i)_aug_\d+$', '', stem)
        by_group[group].append((m, label))
    # Keep groups with one unambiguous folder label; choose one clip per group.
    groups = {g: rows for g, rows in by_group.items() if len({label for _, label in rows}) == 1}
    per_class = {c: [g for g, rows in groups.items() if rows[0][1] == c] for c in (0, 1)}
    if min(map(len, per_class.values())) < 12:
        raise RuntimeError(f'Not enough distinct filename groups for a pilot: real={len(per_class[0])}, fake={len(per_class[1])}')
    chosen = {}
    for c in (0, 1):
        names = sorted(per_class[c]); random.shuffle(names)
        chosen.update({g: groups[g][0] for g in names[:GROUPS_PER_CLASS]})
    print(f'Archive MP4 entries: {len(members)}; candidate groups: real={len(per_class[0])}, fake={len(per_class[1])}')
    print(f'Run selected {len(chosen)} videos across {sum(1 for _, y in chosen.values() if y == 0)} real and {sum(1 for _, y in chosen.values() if y == 1)} fake filename groups.')
    samples = []
    for group, (info, label) in chosen.items():
        target = WORK / 'videos' / str(label) / (hashlib.sha1(group.encode()).hexdigest() + '.mp4')
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            with zf.open(info) as src, target.open('wb') as dst: shutil.copyfileobj(src, dst, 1024 * 1024)
        samples.append({'group_id': group, 'label': label, 'archive_path': info.filename, 'video_path': str(target)})
print('Selected ZIP members read and CRC-checked:', len(samples))

In [ ]:
# Extract up to eight frames per selected video and split by inferred source group.
import cv2
from sklearn.model_selection import GroupShuffleSplit
frames = []
for row in samples:
    cap = cv2.VideoCapture(row['video_path'])
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps = cap.get(cv2.CAP_PROP_FPS) or 1
    if n <= 0:
        cap.release(); continue
    indices = sorted(set(round(i * max(1, n - 1) / max(1, FRAMES_PER_VIDEO - 1)) for i in range(FRAMES_PER_VIDEO)))
    folder = WORK / 'frames' / str(row['label']) / hashlib.sha1(row['group_id'].encode()).hexdigest()
    folder.mkdir(parents=True, exist_ok=True)
    for j, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, image = cap.read()
        if ok:
            path = folder / f'{j:02d}.jpg'; cv2.imwrite(str(path), image)
            frames.append({'path': str(path), 'label': row['label'], 'group_id': row['group_id']})
    cap.release()
if len({r['label'] for r in frames}) != 2: raise RuntimeError('Frame extraction did not produce both classes.')
# Try deterministic group splits until every split contains both labels.
import numpy as np
groups = np.array([r['group_id'] for r in frames]); y = np.array([r['label'] for r in frames])
split_ix = None
for seed in range(SEED, SEED + 200):
    outer = GroupShuffleSplit(n_splits=1, test_size=.30, random_state=seed)
    tr, rest = next(outer.split(np.zeros(len(y)), y, groups))
    inner = GroupShuffleSplit(n_splits=1, test_size=.50, random_state=seed + 1)
    va_rel, te_rel = next(inner.split(np.zeros(len(rest)), y[rest], groups[rest]))
    va, te = rest[va_rel], rest[te_rel]
    if all(len(set(y[ix])) == 2 for ix in (tr, va, te)):
        split_ix = {'train': tr, 'validation': va, 'test': te}; break
if split_ix is None: raise RuntimeError('Could not make three group-disjoint splits with both classes; increase GROUPS_PER_CLASS.')
print('Extracted frames:', len(frames), {k: len(v) for k, v in split_ix.items()})
print('Unique groups:', {k: len(set(groups[v])) for k, v in split_ix.items()})

In [ ]:
# GPU-backed pretrained ResNet-18; validation selects the checkpoint and threshold.
import copy, datetime, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

if not torch.cuda.is_available():
    raise RuntimeError("No GPU is active. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run this notebook from the top.")

device = torch.device("cuda")
print("Training device:", torch.cuda.get_device_name(0))
tfm = transforms.Compose([
    transforms.Resize((160,160)),
    transforms.ToTensor(),
    transforms.Normalize((.485,.456,.406),(.229,.224,.225))
])

class Frames(Dataset):
    def __init__(self, indices): self.rows = [frames[i] for i in indices]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        with Image.open(r["path"]) as im: x = tfm(im.convert("RGB"))
        return x, int(r["label"]), r["group_id"]

train_ix = split_ix["train"]
train_y = np.array([frames[i]["label"] for i in train_ix])
class_counts = np.bincount(train_y, minlength=2)
if min(class_counts) == 0: raise RuntimeError(f"Training split missing a class: {class_counts.tolist()}")
sample_weights = torch.DoubleTensor([1.0 / class_counts[y] for y in train_y])
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_y), replacement=True)
loaders = {
    "train": DataLoader(Frames(train_ix), batch_size=32, sampler=sampler, num_workers=2, pin_memory=True),
    "validation": DataLoader(Frames(split_ix["validation"]), batch_size=32, shuffle=False, num_workers=2, pin_memory=True),
    "test": DataLoader(Frames(split_ix["test"]), batch_size=32, shuffle=False, num_workers=2, pin_memory=True),
}

weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=.3, patience=1)
history, best_state, best_val_loss, best_epoch, stale = [], None, float("inf"), 0, 0
PATIENCE = 3

def evaluate(loader):
    model.eval()
    loss_sum = 0.0; seen = 0; scores_by_group = defaultdict(list); labels_by_group = {}
    with torch.no_grad():
        for x, yb, group_ids in loader:
            x, yb = x.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            logits = model(x)
            loss_sum += criterion(logits, yb).item() * len(yb); seen += len(yb)
            scores = torch.softmax(logits, 1)[:, 1].cpu().tolist()
            for group_id, label, score in zip(group_ids, yb.cpu().tolist(), scores):
                if group_id in labels_by_group and labels_by_group[group_id] != label:
                    raise RuntimeError(f"Conflicting labels in group {group_id}")
                labels_by_group[group_id] = int(label)
                scores_by_group[group_id].append(float(score))
    ordered = sorted(scores_by_group)
    group_probs = np.asarray([np.mean(scores_by_group[g]) for g in ordered])
    group_truth = np.asarray([labels_by_group[g] for g in ordered])
    return loss_sum / max(1, seen), group_probs, group_truth

for epoch in range(EPOCHS):
    model.train(); loss_sum = 0.0; correct = seen = 0
    for x, yb, _group_ids in loaders["train"]:
        x, yb = x.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x); loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        loss_sum += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); seen += len(yb)
    val_loss, _, _ = evaluate(loaders["validation"])
    scheduler.step(val_loss)
    row = {
        "epoch": epoch + 1,
        "train_loss": loss_sum / max(1, seen),
        "train_accuracy": correct / max(1, seen),
        "validation_loss": val_loss,
        "learning_rate": optimizer.param_groups[0]["lr"],
    }
    history.append(row); print(row)
    if val_loss < best_val_loss:
        best_val_loss, best_epoch, stale = val_loss, epoch + 1, 0
        best_state = copy.deepcopy(model.state_dict())
    else:
        stale += 1
        if stale >= PATIENCE:
            print("Early stopping: validation loss did not improve.")
            break

if best_state is None: raise RuntimeError("Training did not produce a checkpoint.")
model.load_state_dict(best_state)
val_loss, val_probs, val_truth = evaluate(loaders["validation"])
fpr, tpr, thresholds = roc_curve(val_truth, val_probs)
threshold_idx = int(np.argmax(tpr - fpr))
chosen_threshold = float(thresholds[threshold_idx])
if not np.isfinite(chosen_threshold): chosen_threshold = 0.5

test_loss, test_probs, test_truth = evaluate(loaders["test"])
test_pred = (test_probs >= chosen_threshold).astype(int)
test_pred_05 = (test_probs >= 0.5).astype(int)
split_summary = {}
for name, ix in split_ix.items():
    split_summary[name] = {
        "frames": int(len(ix)),
        "groups": int(len(set(groups[ix]))),
        "real_frames": int(sum(y[ix] == 0)),
        "synthetic_frames": int(sum(y[ix] == 1)),
        "real_groups": int(len(set(groups[ix][y[ix] == 0]))),
        "synthetic_groups": int(len(set(groups[ix][y[ix] == 1]))),
    }

report = {
    "run_type": "exploratory_pretrained_gpu_baseline_not_validated",
    "dataset": "SDFVD2.0 v1",
    "seed": SEED,
    "groups_per_class_cap": GROUPS_PER_CLASS,
    "frames_per_video_cap": FRAMES_PER_VIDEO,
    "max_epochs": EPOCHS,
    "best_epoch": best_epoch,
    "device": torch.cuda.get_device_name(0),
    "label_source": "candidate inferred from archive folder name",
    "group_source": "candidate inferred by removing trailing _aug_N from filename",
    "threshold_selection": "max validation Youden J on group-mean scores; test split untouched until final evaluation",
    "evaluation_unit": "one mean score per inferred source group; frames are not counted as independent test samples",
    "threshold": chosen_threshold,
    "split_summary": split_summary,
    "history": history,
    "validation_loss": val_loss,
    "test_loss": test_loss,
    "test_confusion_matrix_at_selected_threshold": confusion_matrix(test_truth, test_pred).tolist(),
    "test_confusion_matrix_at_0_5": confusion_matrix(test_truth, test_pred_05).tolist(),
    "test_classification_report_at_selected_threshold": classification_report(test_truth, test_pred, output_dict=True, zero_division=0),
    "test_roc_auc": float(roc_auc_score(test_truth, test_probs)) if len(set(test_truth)) == 2 else None,
    "test_group_count": int(len(test_truth)),
    "test_frame_count": int(len(split_ix["test"])) ,
    "limitations": [
        "exploratory subset, not a final benchmark",
        "candidate labels and filename groups not human-reviewed",
        "cross-class source identity not verified",
        "small number of independent source groups",
    ],
}
run_id = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
model_path = OUT / f"sdfvd2_resnet18_{run_id}.pt"
report_path = OUT / f"sdfvd2_report_{run_id}.json"
torch.save(best_state, model_path)
report_path.write_text(json.dumps(report, indent=2))
print("Run:", run_id, "| best epoch:", best_epoch, "| validation threshold:", chosen_threshold)
print("Split summary:", json.dumps(split_summary, indent=2))
print("Test confusion matrix at selected threshold:", report["test_confusion_matrix_at_selected_threshold"])

folder_q = "name = 'Deepfake pilot outputs' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
folders = drive_api.files().list(q=folder_q, fields="files(id,name)", pageSize=20).execute().get("files", [])
if folders:
    output_folder_id = folders[0]["id"]
else:
    output_folder_id = drive_api.files().create(
        body={"name":"Deepfake pilot outputs","mimeType":"application/vnd.google-apps.folder"},
        fields="id"
    ).execute()["id"]
uploaded = []
for path, mime in [(model_path, "application/octet-stream"), (report_path, "application/json")]:
    item = drive_api.files().create(
        body={"name":path.name, "parents":[output_folder_id]},
        media_body=MediaFileUpload(str(path), mimetype=mime, resumable=True),
        fields="id,name,webViewLink"
    ).execute()
    uploaded.append(item)
print("Uploaded checkpoint and report to Drive:", uploaded)
